# Feature — Docs-to-Multi-Hop-QA Synthesis Pipeline (rag/synthesis)

The synthesis package: docs -> typed entity graph -> multi-hop path sampling -> answer-first QA generation -> chain-dependency verification -> 2D difficulty labeling/balance -> a GRPO-ready train/val split. The CLI hardcodes a Groq client (no key available here); this runs the SAME run_pipeline() with a real local OpenAI-compatible LLM client instead (server backed by real Qwen2.5-3B-Instruct) -- no fake/mock client, only the LLM transport target changes.


In [1]:
"""
Feature — Docs-to-Multi-Hop-QA Synthesis Pipeline (agenttune.rag.synthesis).

This is the synthesis package: docs -> typed entity graph -> multi-hop path
sampling -> answer-first QA generation -> chain-dependency verification -> 2D
difficulty labeling/balance -> a GRPO-ready train/val split (gold-chunk-disjoint).
Its own README claims "VALIDATED end-to-end on real models" (BGE-M3 + Qwen3.6).

`build_dataset.py`'s CLI `main()` hardcodes a `GroqLLMClient`, which needs a
GROQ_API_KEY not available in this environment. The LLM client is injectable
(`LLMClient` Protocol) specifically so the pipeline can run against any real
OpenAI-compatible endpoint (`OpenAICompatLLMClient` — "vLLM, TGI,
Together, local LLM servers, etc.", per its own docstring). So instead of a
fake/mock client, this script stands up a real local OpenAI-compatible HTTP
server backed by a real local `Qwen2.5-3B-Instruct` (same pattern as
`self_heal_llm_real.py`) and points a real `OpenAICompatLLMClient` at it, then
calls the SAME `run_pipeline()` the CLI calls — no pipeline logic is bypassed,
only the LLM *transport* target changes (local server instead of Groq's).

Scale is reduced from the CLI's defaults (train_size=50, per_hop=30) to keep
this a fast, honest, real run: real Stage 0-5 execution over a small real
corpus, not a claim about dataset-scale throughput.
"""
import os
os.environ.setdefault("HF_HUB_OFFLINE", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OPENAI_API_KEY", "sk-local-noop")

'sk-local-noop'

In [2]:
import json
import tempfile
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
from agenttune.rag.data.hotpotqa import load_hotpotqa_splits, build_corpus_from_hotpotqa
from agenttune.rag.synthesis.build_dataset import run_pipeline
from agenttune.rag.synthesis.llm_client import OpenAICompatLLMClient
from agenttune.rag.synthesis.embeddings import BGEM3Embedder

In [5]:
JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

In [6]:
def start_llm_server():
    tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype=torch.bfloat16).to("cuda")

    def generate(messages, temperature=0.0, max_new_tokens=300):
        enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True).to("cuda")
        sample = bool(temperature and temperature > 0)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=sample,
                                  temperature=temperature if sample else None,
                                  top_p=0.95 if sample else None,
                                  pad_token_id=tok.pad_token_id or tok.eos_token_id)
        return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *a):
            pass

        def do_POST(self):
            body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
            text = generate(body["messages"], body.get("temperature", 0.0))
            payload = json.dumps({
                "choices": [{"index": 0, "message": {"role": "assistant", "content": text},
                             "finish_reason": "stop"}],
                "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
            }).encode()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(payload)))
            self.end_headers()
            self.wfile.write(payload)

    server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, server.server_address[1]

In [7]:
def main():
    print(f"[gpu]    {torch.cuda.get_device_name(0)}  (cuda: {torch.cuda.is_available()})")

    server, port = start_llm_server()
    api_base = f"http://127.0.0.1:{port}/v1"
    print(f"[server] real {JUDGE_MODEL} on {api_base} — a real local OpenAI-compatible endpoint, not a mock")

    llm = OpenAICompatLLMClient(model=f"openai/{JUDGE_MODEL}", api_key="sk-local-noop", base_url=api_base)

    print("[embed]  loading real BAAI/bge-m3 embedder (this is the same embedder the package's README validates against)...")
    embedder = BGEM3Embedder(device="cuda")

    print("[corpus] loading a small real HotpotQA slice...")
    train, _ = load_hotpotqa_splits(train_size=5, eval_size=0)
    docs = build_corpus_from_hotpotqa(train, max_docs=20)
    print(f"[corpus] {len(docs)} real documents")

    out_dir = tempfile.mkdtemp(prefix="agenttune_rag_synthesis_")
    print(f"[out_dir] {out_dir}")

    samples = run_pipeline(
        docs=docs, llm=llm, embedder=embedder, out_dir=out_dir,
        backend="sqlite", per_hop=3, target_per_cell=2,
        val_fraction=0.3, split_level="chunk", use_cache=False,
    )

    print(f"\n[result] {len(samples)} real multi-hop QA samples generated end-to-end")
    for s in samples[:3]:
        print(f"  Q: {s.question}")
        print(f"  A: {s.answer}")
        print(f"  hops: {s.hop_count}  difficulty_cell: {s.difficulty_cell}")
    print(f"\n[files] {sorted(os.listdir(out_dir))}")

In [8]:
if __name__ == "__main__":
    main()

[gpu]    NVIDIA L4  (cuda: True)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[server] real Qwen/Qwen2.5-3B-Instruct on http://127.0.0.1:54467/v1 — a real local OpenAI-compatible endpoint, not a mock


[embed]  loading real BAAI/bge-m3 embedder (this is the same embedder the package's README validates against)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[corpus] loading a small real HotpotQA slice...


[corpus] 20 real documents
[out_dir] /tmp/agenttune_rag_synthesis_wjyu42k3


[stage0] built + cached: 31 chunks, 93 edges (use_cache=False)



[result] 4 real multi-hop QA samples generated end-to-end
  Q: In which city does the Collard Festival take place?
  A: Greenville
  hops: 2  difficulty_cell: 2hop_easy
  Q: Who produced the animated films 'An American Tail: Fievel Goes West' and 'Balto'?
  A: Steven Spielberg
  hops: 2  difficulty_cell: 2hop_easy
  Q: What city is the Greenville Metropolitan Area located in?
  A: Greenville
  hops: 3  difficulty_cell: 3hop_easy

[files] ['corpus.sqlite', 'cost_summary.json', 'dataset_all.csv', 'dataset_all.parquet', 'dataset_final.csv', 'dataset_final.parquet', 'dataset_grpo.jsonl', 'dataset_train.csv', 'dataset_train.parquet', 'dataset_train_grpo.jsonl', 'dataset_val.csv', 'dataset_val_grpo.jsonl', 'manifest.json', 'metrics.json', 'stage0_cache.pkl', 'stage0_chunks.csv', 'stage0_chunks.parquet', 'stage0_graph_edges.csv', 'stage0_graph_edges.parquet', 'stage1_paths.csv', 'stage1_paths.parquet', 'stage2_generated.csv', 'stage2_generated.parquet', 'stage3_verified.csv', 'stage3_verifie